# repo-code-completion: GPU demo (Ollama + qwen2.5-coder:7b)

Runs the indexer -> BM25/Symbol/Dependency retrieval -> LLM selection pipeline against a GPU-accelerated local Ollama server, instead of CPU-only.

**Before running**: Runtime -> Change runtime type -> select a GPU (T4 is fine).

## 1. Install and start Ollama, pull the model

In [ ]:
import subprocess, time, shutil, os

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "zstd"], check=True)

install_result = subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True, capture_output=True, text=True,
)
print(install_result.stdout)
print(install_result.stderr)
install_result.check_returncode()  # raise here with the real error if the installer failed

# The installer usually puts the binary in /usr/local/bin, which may not be on
# this kernel's PATH yet even though the install itself succeeded.
ollama_path = shutil.which("ollama") or next(
    (p for p in ["/usr/local/bin/ollama", "/usr/bin/ollama"] if os.path.exists(p)), None
)
if not ollama_path:
    raise RuntimeError("ollama binary not found after install -- see the installer output printed above")
print("ollama found at:", ollama_path)

subprocess.Popen([ollama_path, "serve"])
time.sleep(5)

pull_result = subprocess.run([ollama_path, "pull", "qwen2.5-coder:7b"], capture_output=True, text=True)
print(pull_result.stdout)
print(pull_result.stderr)
pull_result.check_returncode()

In [ ]:
# Confirm it's up and (ideally) running on GPU, not CPU.
!curl -s http://localhost:11434/api/tags | head -c 300
print()
subprocess.run([ollama_path, "ps"])

## 2. Get the project code onto Colab

Two options -- use whichever applies:

- **Option A (if you've pushed this repo to GitHub)**: set `REPO_URL` below and run the cell.
- **Option B (no GitHub remote yet)**: zip your local `repo-code-completion` folder, then run the *next* cell instead -- it will prompt you to upload the zip.

In [ ]:
# Option A: clone from GitHub (skip this cell if you're using Option B).
REPO_URL = "https://github.com/Robertkiza0/repo-code.git"
if REPO_URL:
    !git clone "$REPO_URL" repo-code-completion

In [ ]:
# Option B: upload a zip of your local repo-code-completion folder.
import os, zipfile

if not os.path.exists("repo-code-completion"):
    from google.colab import files
    print("Zip your local repo-code-completion folder, then upload it here.")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(".")
    # if the zip's top-level entry isn't already named repo-code-completion,
    # find the extracted folder and rename it to match.
    if not os.path.exists("repo-code-completion"):
        extracted = [n.split("/")[0] for n in zipfile.ZipFile(zip_name).namelist()]
        top = sorted(set(extracted))[0]
        os.rename(top, "repo-code-completion")

In [ ]:
%cd repo-code-completion

## 3. Install only what this pipeline needs

(Skips the heavy generation-stage deps in `requirements.txt` like `torch`/`vllm` -- not needed for indexing/retrieval/selection.)

In [ ]:
!pip install -q tree-sitter tree-sitter-python tree-sitter-java tree-sitter-typescript tree-sitter-c-sharp rank-bm25 requests

## 4. Run the pipeline: index -> retrieve -> select

In [ ]:
from indexer.repo_parser import RepoParser
from retrieval.bm25_retriever import BM25Retriever
from retrieval.symbol_retriever import SymbolRetriever
from retrieval.dependency_retriever import DependencyRetriever
from retrieval.candidate_pipeline import CandidatePipeline
from selection.llm_selector import LLMSelector

chunks = [c.to_dict() for c in RepoParser("tests/sample_repo").parse_repo()]
pipeline = CandidatePipeline(BM25Retriever(chunks), SymbolRetriever(chunks), DependencyRetriever(chunks))

code_before_cursor = "result = Greeter("
target_file = "pkg/module_b.py"

candidates = pipeline.nominate(code_before_cursor, target_file=target_file)
print(f"{len(candidates)} candidates:")
for c in candidates:
    print(f"  {c['name']:12s} sources={c['sources']} scores={c['scores']}")

In [ ]:
import time

selector = LLMSelector(chunks)  # defaults to localhost:11434, qwen2.5-coder:7b
t0 = time.time()
result = selector.select(code_before_cursor, target_file, candidates)
print(f"took {time.time() - t0:.1f}s (should be seconds on GPU, not minutes)")
print()
print("selected_chunk_ids:         ", result["selected_chunk_ids"])
print("candidate_chunk_ids:        ", result["candidate_chunk_ids"])
print("rejected_hallucinated_ids:  ", result["rejected_hallucinated_ids"])

## 5. Alternative: Hugging Face backend instead of Ollama

Runs `Qwen/Qwen2.5-Coder-7B-Instruct` in-process via `transformers`, on the same
GPU, with no separate server/install step -- useful if the Ollama install
above is being finicky, or you'd rather not run a background service in a
notebook. Uses the same `chunks`/`candidates`/`code_before_cursor`/`target_file`
from section 4 above.

**Needs a Hugging Face access token** (create one at
https://huggingface.co/settings/tokens if you don't have one) -- add it as a
Colab Secret named `HF_TOKEN` (padlock icon, left sidebar) before running the
login cell below, so it's never pasted directly into the notebook.

In [ ]:
!pip install -q torch transformers accelerate

In [ ]:
# Log in with your HF access token, without ever typing/committing it into a
# cell. Preferred: Colab's Secrets manager (padlock icon in the left sidebar)
# -- add a secret named HF_TOKEN there and grant this notebook access.
# Falls back to a hidden interactive prompt if no HF_TOKEN secret is set.
from huggingface_hub import login

try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    login()  # prompts for the token interactively (input is hidden)

In [ ]:
import time
from selection.backends import HuggingFaceBackend
from selection.llm_selector import LLMSelector

hf_backend = HuggingFaceBackend()  # downloads Qwen2.5-Coder-7B-Instruct on first run
hf_selector = LLMSelector(chunks, backend=hf_backend)

t0 = time.time()
hf_result = hf_selector.select(code_before_cursor, target_file, candidates)
print(f"took {time.time() - t0:.1f}s")
print()
print("selected_chunk_ids:         ", hf_result["selected_chunk_ids"])
print("candidate_chunk_ids:        ", hf_result["candidate_chunk_ids"])
print("rejected_hallucinated_ids:  ", hf_result["rejected_hallucinated_ids"])